In [1]:
# =============================================================================
# TAHAP 0: IMPORT LIBRARY
# =============================================================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb  # Ditambahkan untuk blending
import optuna          # Ditambahkan untuk optimisasi
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')
print("✅ Library berhasil diimpor.")

# =============================================================================
# TAHAP 1: PERSIAPAN DATA DASAR
# =============================================================================
print("\n--- Tahap 1: Memuat dan Mempersiapkan Data ---")
# Ganti dengan path ke folder dataset Anda
DATA_PATH = '../dataset/'

try:
    train_trans_df = pd.read_csv(f'{DATA_PATH}train_transaction_data.csv')
    test_trans_df = pd.read_csv(f'{DATA_PATH}test_transaction_data.csv')
    prodgram_df = pd.read_csv(f'{DATA_PATH}prodgram_data.csv')
    train_label_df = pd.read_csv(f'{DATA_PATH}train_label_data.csv')
    # Membuat kerangka submission dari data tes untuk memastikan semua MemberID ter-cover
    sample_submission_df = pd.read_csv(f'{DATA_PATH}test_transaction_data.csv')[['MemberID']].drop_duplicates()
    print("✅ Semua file berhasil dimuat.")
except FileNotFoundError as e:
    print(f"❌ Error: File tidak ditemukan. Pastikan path '{DATA_PATH}' sudah benar. Detail: {e}")
    exit()

# =============================================================================
# TAHAP 2: REKAYASA FITUR BERBASIS SIKLUS KONSUMSI
# =============================================================================
print("\n--- Tahap 2: Rekayasa Fitur Berbasis Konsumsi ---")

# Ekstrak nilai gramasi numerik dari nama
prodgram_df['GrammageNumeric'] = prodgram_df['GrammageName'].str.extract('(\d+)').astype(float)
prodgram_df['GrammageNumeric'].fillna(prodgram_df['GrammageNumeric'].median(), inplace=True)

# Gabungkan semua data transaksi
all_trans_df = pd.concat([train_trans_df, test_trans_df], ignore_index=True)
all_trans_df['TransactionDatetime'] = pd.to_datetime(all_trans_df['TransactionDatetime']).dt.tz_localize(None)

# Gabungkan dengan data prodgram untuk mendapatkan gramasi
all_trans_df = pd.merge(all_trans_df, prodgram_df[['prodgramID', 'GrammageNumeric']], left_on='FK_PROD_GRAM_ID', right_on='prodgramID', how='left')

# Hitung 'TotalGrammage' sebagai sinyal konsumsi utama
all_trans_df['TotalGrammage'] = all_trans_df['Qty'] * all_trans_df['GrammageNumeric']
all_trans_df['Month'] = all_trans_df['TransactionDatetime'].dt.to_period('M')
print("✅ Fitur 'TotalGrammage' sebagai sinyal konsumsi telah dibuat.")

# Membuat Agregat Bulanan
monthly_agg = all_trans_df.groupby(['MemberID', 'Month']).agg(
    TotalGrammage=('TotalGrammage', 'sum'),
    Frequency=('TransactionID', 'nunique')
).reset_index()
print("✅ Agregat konsumsi & frekuensi bulanan telah dibuat.")

# Membuat "Kanvas" Data Latih (Point-in-Time)
all_months = monthly_agg['Month'].unique()
all_members = all_trans_df['MemberID'].unique()
canvas_df = pd.MultiIndex.from_product([all_members, all_months], names=['MemberID', 'Month']).to_frame(index=False)
data = pd.merge(canvas_df, monthly_agg, on=['MemberID', 'Month'], how='left').fillna(0)
data = data.sort_values(by=['MemberID', 'Month'])

# Membuat Lag & Rolling Features
features_to_lag = ['TotalGrammage', 'Frequency']
lags = [1, 2, 3]
windows = [3, 6]

for feat in features_to_lag:
    for lag in lags:
        data[f'{feat}_lag_{lag}'] = data.groupby('MemberID')[feat].shift(lag).fillna(0)
    for window in windows:
        data[f'{feat}_rolling_mean_{window}'] = data.groupby('MemberID')[feat].shift(1).rolling(window, min_periods=1).mean().fillna(0)
        data[f'{feat}_rolling_std_{window}'] = data.groupby('MemberID')[feat].shift(1).rolling(window, min_periods=1).std().fillna(0)
print("✅ Fitur Lag dan Rolling Window telah dibuat.")

# Membuat Label Target
data['target'] = data.groupby('MemberID')['TotalGrammage'].shift(-1).fillna(0)
data['target'] = (data['target'] > 0).astype(int)
# Hapus bulan terakhir dari setiap member karena tidak memiliki target
data_final = data.groupby('MemberID').head(-1).copy()
print("✅ Label target telah dibuat.")

# =============================================================================
# TAHAP 3: PERSIAPAN DATA LATIH DAN TES FINAL
# =============================================================================
print("\n--- Tahap 3: Mempersiapkan Data Latih dan Tes ---")

# Data Latih: Semua data historis dari member yang ada di train_label_df
train_member_ids = train_label_df['MemberID'].unique()
X_full_train = data_final[data_final['MemberID'].isin(train_member_ids)].copy()
y_full_train = X_full_train['target']

# Data Tes: Fitur dari bulan terakhir untuk setiap member
latest_features_df = data.loc[data.groupby('MemberID')['Month'].idxmax()]
X_test_final = pd.merge(sample_submission_df[['MemberID']], latest_features_df, on='MemberID', how='left').fillna(0)

# Definisikan kolom fitur
features = [col for col in data_final.columns if col not in ['MemberID', 'Month', 'TotalGrammage', 'Frequency', 'target']]
X_full_train = X_full_train[features]
X_test_final = X_test_final[features]
print("✅ Data Latih dan Tes siap untuk pemodelan.")

<>:41: SyntaxWarning: invalid escape sequence '\d'
<>:41: SyntaxWarning: invalid escape sequence '\d'
C:\Users\MSI Modern 14\AppData\Local\Temp\ipykernel_1724\1784478214.py:41: SyntaxWarning: invalid escape sequence '\d'
  prodgram_df['GrammageNumeric'] = prodgram_df['GrammageName'].str.extract('(\d+)').astype(float)


✅ Library berhasil diimpor.

--- Tahap 1: Memuat dan Mempersiapkan Data ---
✅ Semua file berhasil dimuat.

--- Tahap 2: Rekayasa Fitur Berbasis Konsumsi ---
✅ Fitur 'TotalGrammage' sebagai sinyal konsumsi telah dibuat.
✅ Agregat konsumsi & frekuensi bulanan telah dibuat.
✅ Fitur Lag dan Rolling Window telah dibuat.
✅ Label target telah dibuat.

--- Tahap 3: Mempersiapkan Data Latih dan Tes ---
✅ Data Latih dan Tes siap untuk pemodelan.


In [2]:
# =============================================================================
# TAHAP 4: PELATIHAN MODEL FINAL (MODEL PERTAMA: LIGHTGBM)
# =============================================================================
print("\n--- Tahap 4: Melatih Model 1 (LightGBM) ---")

# Hitung bobot kelas untuk menangani data tidak seimbang
scale_pos_weight = y_full_train.value_counts()[0] / y_full_train.value_counts()[1]

# Inisialisasi dan latih model final LightGBM
# Di sini Anda bisa menambahkan parameter terbaik dari Optuna
final_lgbm = lgb.LGBMClassifier(
    objective='binary',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)
final_lgbm.fit(X_full_train, y_full_train)
print("✅ Model LightGBM final berhasil dilatih.")

# Dapatkan prediksi probabilitas dari LightGBM
test_preds_lgbm = final_lgbm.predict_proba(X_test_final)[:, 1]
print("✅ Prediksi dari LightGBM berhasil didapatkan.")


--- Tahap 4: Melatih Model 1 (LightGBM) ---
✅ Model LightGBM final berhasil dilatih.
✅ Prediksi dari LightGBM berhasil didapatkan.


In [4]:
# =============================================================================
# TAHAP 5: PELATIHAN MODEL FINAL (MODEL KEDUA: CATBOOST)
# =============================================================================
print("\n--- Tahap 5: Melatih Model 2 (CatBoost) ---")

# Inisialisasi model CatBoost
# 'auto_class_weights' adalah cara CatBoost menangani data tidak seimbang
final_catboost = cb.CatBoostClassifier(
    iterations=1500,          # Parameter ini bisa dioptimalkan dengan Optuna
    learning_rate=0.05,       # Parameter ini bisa dioptimalkan dengan Optuna
    depth=6,                  # Parameter ini bisa dioptimalkan dengan Optuna
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=0                 # Set ke 100 untuk melihat progres pelatihan
)

# Latih model pada seluruh data latih
final_catboost.fit(X_full_train, y_full_train)
print("✅ Model CatBoost final berhasil dilatih.")

# Dapatkan prediksi probabilitas dari CatBoost
test_preds_catboost = final_catboost.predict_proba(X_test_final)[:, 1]
print("✅ Prediksi dari CatBoost berhasil didapatkan.")


--- Tahap 5: Melatih Model 2 (CatBoost) ---
✅ Model CatBoost final berhasil dilatih.
✅ Prediksi dari CatBoost berhasil didapatkan.


In [7]:
# =============================================================================
# TAHAP 6: OPTIMISASI THRESHOLD DAN SUBMISI
# =============================================================================
print("\n--- Tahap 6: Mencari Threshold Optimal dan Membuat Submisi ---")

# --- Mencari Threshold Optimal menggunakan Cross-Validation pada Data Latih ---
print("⏳ Mencari threshold optimal...")
oof_preds_lgbm = np.zeros(len(X_full_train))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # 5-fold untuk kecepatan

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_train, y_full_train)):
    X_train_fold, y_train_fold = X_full_train.iloc[train_idx], y_full_train.iloc[train_idx]
    X_val_fold, y_val_fold = X_full_train.iloc[val_idx], y_full_train.iloc[val_idx]
    
    # Latih model sementara pada fold ini
    temp_lgbm = lgb.LGBMClassifier(objective='binary', scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1)
    temp_lgbm.fit(X_train_fold, y_train_fold)
    
    # Simpan prediksi pada data validasi (out-of-fold)
    oof_preds_lgbm[val_idx] = temp_lgbm.predict_proba(X_val_fold)[:, 1]

# Cari F1-score untuk setiap kemungkinan threshold
thresholds = np.arange(0.01, 0.51, 0.01)
f1_scores = [f1_score(y_full_train, (oof_preds_lgbm > t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1_score = np.max(f1_scores)

# Hitung balanced accuracy pada threshold terbaik yang ditemukan berdasarkan F1-score
best_balanced_acc = balanced_accuracy_score(y_full_train, (oof_preds_lgbm > best_threshold).astype(int))

print(f"✅ Threshold optimal ditemukan: {best_threshold:.2f}")
print(f"   - Skor F1 terbaik pada data validasi: {best_f1_score:.4f}")
print(f"   - Skor Balanced Accuracy pada threshold tersebut: {best_balanced_acc:.4f}")


# --- Blending Prediksi ---
blended_preds_proba = (0.5 * test_preds_lgbm) + (0.5 * test_preds_catboost)

# --- Membuat DataFrame Submisi ---
submission_df = pd.DataFrame({
    'MemberID': sample_submission_df['MemberID'],
    'next_buy': (blended_preds_proba > best_threshold).astype(int)
})

# Simpan ke file CSV
submission_df.to_csv('../submission/submission-test-001.csv', index=False)

print(f"\n🎉 File 'submission_blending_final.csv' telah berhasil dibuat!")
print(f"Jumlah prediksi 'Beli' (1) dengan threshold {best_threshold:.2f}: {submission_df['next_buy'].sum()}")
print("\nContoh hasil submisi:")
print(submission_df.head())


--- Tahap 6: Mencari Threshold Optimal dan Membuat Submisi ---
⏳ Mencari threshold optimal...
✅ Threshold optimal ditemukan: 0.50
   - Skor F1 terbaik pada data validasi: 0.3719
   - Skor Balanced Accuracy pada threshold tersebut: 0.6351

🎉 File 'submission_blending_final.csv' telah berhasil dibuat!
Jumlah prediksi 'Beli' (1) dengan threshold 0.50: 2997

Contoh hasil submisi:
                           MemberID  next_buy
0  c2a630e3d0dc77dac0f63424a9ae1438         0
1  3ecf7484c08418953e967a20de37051b         1
2  97bbd6c99a862f20657d9b2b1c77b2c8         1
3  3ce072ff9c6f2f4b7c95dbc08324a24d         0
4  ab0b0de2a1c85a40b5d58644aef745c0         0
